In [0]:
import io
import random
from datetime import datetime, UTC

import pandas as pd


# ==========================================
# Municípios reais (amostra ampliada)
# ==========================================

MUNICIPIOS = [
    "1100015", "1100023", "1100031", "1100049",
    "1200013", "1200054", "1300029", "1302603",
    "1500107", "1501402", "1600105", "1702109",
    "2100055", "2207702", "2304400", "2408102",
    "2507507", "2607901", "2704302", "2800308",
    "2905701", "2910800", "2927408", "3101706",
    "3118601", "3136702", "3154606", "3169901",
    "3205309", "3304557", "3509502", "3529401",
    "4106902", "4202008", "4216602", "4305108",
    "4314902", "5002704", "5103403", "5104807",
    "5208707", "5300108"
]

# ==========================================
# Domínios reais
# ==========================================

CADERNOS = [1, 2]

SERIES = [2, 3]

REDES = [2, 3, 5]

PRESENCA = [0, 1]

PREENCHIMENTO_CADERNO = [0, 1]

ALFABETIZADO = [0, 1]


# ==========================================
# Registro sintético
# ==========================================

def gerar_registro():

    return {

        "ano": 2025,

        "id_municipio": random.choice(
            MUNICIPIOS
        ),

        "id_escola": str(
            random.randint(
                60000000,
                60999999
            )
        ),

        "id_aluno": str(
            random.randint(
                30000000,
                59999999
            )
        ),

        "caderno": str(
            random.choice(
                CADERNOS
            )
        ),

        "serie": str(
            random.choice(
                SERIES
            )
        ),

        "rede": str(
            random.choice(
                REDES
            )
        ),

        "presenca": str(
            random.choice(
                PRESENCA
            )
        ),

        "preenchimento_caderno": str(
            random.choice(
                PREENCHIMENTO_CADERNO
            )
        ),

        "alfabetizado": str(
            random.choice(
                ALFABETIZADO
            )
        ),

        "proficiencia": round(
            random.uniform(
                578.46,
                904.38
            ),
            6
        ),

        "peso_aluno": round(
            random.uniform(
                0.5,
                2.0
            ),
            6
        ),

        "_ingested_at": datetime.now(
            UTC
        ).isoformat(),

        "_source_table": (
            "basedosdados."
            "br_inep_avaliacao_alfabetizacao."
            "alunos"
        )
    }


# ==========================================
# Lote
# ==========================================

def gerar_lote(qtd_registros):

    registros = [
        gerar_registro()
        for _ in range(qtd_registros)
    ]

    return pd.DataFrame(
        registros
    )


# ==========================================
# Upload Parquet
# ==========================================

def upload_parquet(
    blob_service_client,
    container_name,
    dataframe,
    blob_name
):

    parquet_buffer = io.BytesIO()

    dataframe.to_parquet(
        parquet_buffer,
        index=False,
        engine="pyarrow"
    )

    blob_client = (
        blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_name
        )
    )

    blob_client.upload_blob(
        parquet_buffer.getvalue(),
        overwrite=True
    )

    return len(
        dataframe
    )


# ==========================================
# Execução Batch
# ==========================================

def execute_batch(
    blob_service_client,
    container_name,
    records,
    folder
):

    df = gerar_lote(
        records
    )

    data_arquivo = datetime.now(
        UTC
    ).strftime(
        "%Y-%m-%d_%H%M%S"
    )

    blob_name = (
        f"{folder}/"
        f"inep_alunos_"
        f"{data_arquivo}.parquet"
    )

    total_sent = upload_parquet(
        blob_service_client,
        container_name,
        df,
        blob_name
    )

    return {
        "records_sent": total_sent,
        "blob_name": blob_name,
        "preview": df.head()
    }